# Week 07. 동기/비동기와 CRUD API 흐름

7주차는 동기 처리와 비동기 처리의 차이, 그리고 CRUD API의 요청/응답 구조를 다룹니다.
네트워크 다운로드 대신 `sleep`으로 대기 시간을 재현해 오류 없이 비교합니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. 동기 처리

동기 코드는 한 작업이 끝난 뒤 다음 작업을 시작합니다.
단순하지만 대기 시간이 많은 작업에서는 전체 시간이 길어집니다.


In [1]:
import time


def sync_task(name, delay=0.05):
    time.sleep(delay)
    return f"{name} done"


start = time.perf_counter()
sync_results = [sync_task(f"task-{i}") for i in range(5)]
sync_elapsed = time.perf_counter() - start

print(sync_results)
print(f"동기 실행 시간: {sync_elapsed:.3f}초")


['task-0 done', 'task-1 done', 'task-2 done', 'task-3 done', 'task-4 done']
동기 실행 시간: 0.273초


## 2. 비동기 처리

비동기 코드는 기다리는 동안 다른 작업을 함께 진행할 수 있습니다.
Jupyter에서는 top-level `await`를 사용할 수 있습니다.


In [2]:
import asyncio
import time


async def async_task(name, delay=0.05):
    await asyncio.sleep(delay)
    return f"{name} done"


start = time.perf_counter()
async_results = await asyncio.gather(*(async_task(f"task-{i}") for i in range(5)))
async_elapsed = time.perf_counter() - start

print(async_results)
print(f"비동기 실행 시간: {async_elapsed:.3f}초")
print("비동기가 더 빠른가?", async_elapsed < sync_elapsed)


['task-0 done', 'task-1 done', 'task-2 done', 'task-3 done', 'task-4 done']
비동기 실행 시간: 0.052초
비동기가 더 빠른가? True


## 3. CRUD API를 함수로 재현

실제 FastAPI에서는 HTTP 메서드가 함수를 호출합니다.
여기서는 딕셔너리 DB와 함수로 같은 흐름을 표현합니다.


In [3]:
from pydantic import BaseModel, Field


class Item(BaseModel):
    name: str = Field(min_length=1)
    price: float = Field(gt=0)
    is_offer: bool = False


item_db = {}


def create_item(item_id: int, item: Item):
    if item_id in item_db:
        return {"status": 409, "message": "이미 존재하는 ID입니다."}
    item_db[item_id] = item.model_dump() if hasattr(item, "model_dump") else item.dict()
    return {"status": 201, "item": item_db[item_id]}


def read_item(item_id: int):
    if item_id not in item_db:
        return {"status": 404, "message": "상품을 찾을 수 없습니다."}
    return {"status": 200, "item": item_db[item_id]}


print(create_item(1, Item(name="Keyboard", price=79.9, is_offer=True)))
print(read_item(1))
print(read_item(999))


{'status': 201, 'item': {'name': 'Keyboard', 'price': 79.9, 'is_offer': True}}
{'status': 200, 'item': {'name': 'Keyboard', 'price': 79.9, 'is_offer': True}}
{'status': 404, 'message': '상품을 찾을 수 없습니다.'}


## 정리

- 동기는 순차 처리라 이해하기 쉽습니다.
- 비동기는 대기 시간이 많은 작업에서 효율적입니다.
- CRUD API는 결국 입력 검증, 상태 확인, 데이터 변경, 응답 생성의 조합입니다.
